In [1]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Global parameters
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_RATE = 20            # Hz  (ESP32 acquisition rate)
DURATION    = 3600          # seconds  (1 hour recording)
N_SAMPLES   = SAMPLE_RATE * DURATION   # 72,000 timestamps
SEED        = 42
 
rng = np.random.default_rng(SEED)
 
FEATURE_COLS = [
    'temperature', 'heart_rate', 'spo2',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x',  'gyro_y',  'gyro_z',
]
 
 
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
# 2.1  Time vector
# ─────────────────────────────────────────────────────────────────────────────
def make_time_vector(duration: float = DURATION, fs: int = SAMPLE_RATE) -> np.ndarray:
    """
    Return a uniformly-spaced 1-D array of timestamps in seconds.
 
    Shape : (N_SAMPLES,)  →  (72000,)
    Step  : 1/fs = 0.05 s
    """
    return np.linspace(0.0, duration, int(duration * fs), endpoint=False)
 
 
# ─────────────────────────────────────────────────────────────────────────────

In [4]:
# 2.2  Helper oscillators
# ─────────────────────────────────────────────────────────────────────────────
def _sum_of_sines(
    t: np.ndarray,
    components: list[tuple],
    rng: np.random.Generator,
) -> np.ndarray:
    """
    Produce a wandering baseline by superposing low-frequency sinusoids.
 
    Each entry in `components` is  (amplitude, freq_hz)  or
    (amplitude, freq_hz, phase_rad).  When phase is omitted it is
    drawn uniformly from [0, 2π] so every run is distinct.
 
    This approach closely mimics the Perlin-noise heuristic at low
    frequencies while remaining fully reproducible and interpretable.
    """
    out = np.zeros(len(t), dtype=np.float64)
    for entry in components:
        amp, freq = entry[0], entry[1]
        phi = entry[2] if len(entry) > 2 else rng.uniform(0.0, 2 * np.pi)
        out += amp * np.sin(2 * np.pi * freq * t + phi)
    return out
 
 
def _band_limited_noise(
    n: int,
    low_hz: float,
    high_hz: float,
    fs: int,
    rng: np.random.Generator,
    amplitude: float = 1.0,
) -> np.ndarray:
    """
    White noise passed through a 4th-order Butterworth bandpass filter.
    Used for IMU channels where the noise spectrum is non-white in practice
    (sensor resonances, cable coupling, mechanical vibration floor).
    """
    white = rng.standard_normal(n)
    nyq   = fs / 2.0
    low   = np.clip(low_hz  / nyq, 1e-4, 0.999)
    high  = np.clip(high_hz / nyq, 1e-4, 0.999)
    b, a  = butter(4, [low, high], btype='band')
    return amplitude * filtfilt(b, a, white)
 
 
# ─────────────────────────────────────────────────────────────────────────────

In [5]:
# 2.2  Signal generators
# ─────────────────────────────────────────────────────────────────────────────
def generate_temperature(t: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """
    Temp(t) = 37.0  +  0.15·sin(2π·f₁·t + φ₁)
                     +  0.05·sin(2π·f₂·t + φ₂)
                     +  𝒩(0, 0.01²)
 
    f₁ = 1/3600 Hz  →  one circadian half-cycle in the recording window
    f₂ = 2/3600 Hz  →  second harmonic for realistic shape asymmetry
    Clipped to physiological bounds [36.5, 37.5] °C.
    """
    f1 = 1.0 / DURATION        # one full cycle per hour
    f2 = 2.0 / DURATION
    trend = _sum_of_sines(t, [(0.15, f1), (0.05, f2)], rng)
    noise = rng.normal(0.0, 0.01, size=len(t))
    return np.clip(37.0 + trend + noise, 36.5, 37.5)
 
 
def generate_heart_rate(t: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """
    HR(t) = 72  +  wandering trend  +  𝒩(0, 0.5²)
 
    Oscillators are grounded in known HRV frequency bands:
      VLF  (<0.003 Hz)  — thermoregulatory / hormonal
      LF   (0.003–0.04) — sympathetic / baroreflex
      HF   (0.15–0.4)   — respiratory sinus arrhythmia (RSA)
    """
    components = [
        (4.0, 1.0 / 900),     # 15-min slow sympathetic drift (LF lower edge)
        (2.5, 1.0 / 300),     # 5-min baroreflex oscillation
        (1.5, 1.0 /  60),     # 1-min vasomotor wave
        (0.8, 0.25),          # RSA fundamental  (~15 breaths/min)
        (0.4, 0.33),          # RSA harmonic / accessory breathing muscle
    ]
    trend = _sum_of_sines(t, components, rng)
    noise = rng.normal(0.0, 0.5, size=len(t))
    return np.clip(72.0 + trend + noise, 60.0, 100.0)
 
 
def generate_spo2(t: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """
    SpO₂(t) = 98.5  +  wandering trend  +  𝒩(0, 0.03²)
 
    Mechanically coupled to HR via shared respiratory driver (RSA phase
    matched to generate realistic cross-correlation between HR & SpO₂).
    Upper-skewed: the hard ceiling at 100% compresses the distribution.
    """
    # Share the RSA phase with HR for physiological cross-correlation
    rsa_phase = rng.uniform(0.0, 2 * np.pi)
    components = [
        (0.6,  1.0 / 900),
        (0.3,  1.0 / 300),
        (0.15, 0.25, rsa_phase),   # same respiratory phase as HR
    ]
    trend = _sum_of_sines(t, components, rng)
    noise = rng.normal(0.0, 0.03, size=len(t))
    return np.clip(98.5 + trend + noise, 95.0, 100.0)
 
 
def generate_accelerometer(
    t: np.ndarray,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Rest-state accelerometer.
 
    X, Y : micro-movements around 0 g.
            Band-limited noise 0.1–5 Hz (postural micro-sway) + white floor.
    Z    : gravity offset of 1.0 g  +  same noise model.
            The DC offset distinguishes Z from the lateral axes — do NOT
            treat all three channels identically in normalisation.
    """
    n = len(t)
    def _ch(dc=0.0, sway_amp=0.05, white_std=0.02):
        sway  = _band_limited_noise(n, 0.1, 5.0, SAMPLE_RATE, rng, sway_amp)
        white = rng.normal(0.0, white_std, n)
        return dc + sway + white
 
    ax = _ch(dc=0.0,  sway_amp=0.05)
    ay = _ch(dc=0.0,  sway_amp=0.05)
    az = _ch(dc=1.0,  sway_amp=0.04)    # gravity component on Z
    return ax, ay, az
 
 
def generate_gyroscope(
    t: np.ndarray,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Rest-state gyroscope — all axes centred at 0 °/s.
 
    Low-amplitude band-limited noise (0.1–3 Hz) simulates postural sway
    and micro-tremor; white floor represents sensor quantisation noise.
    No DC offset (unlike Accel Z) so Z is statistically exchangeable.
    """
    n = len(t)
    def _ch(sway_amp=1.5, white_std=0.5):
        sway  = _band_limited_noise(n, 0.1, 3.0, SAMPLE_RATE, rng, sway_amp)
        white = rng.normal(0.0, white_std, n)
        return sway + white
 
    gx = _ch(sway_amp=1.5)
    gy = _ch(sway_amp=1.5)
    gz = _ch(sway_amp=0.8)     # yaw has lower postural drift amplitude
    return gx, gy, gz
 
 
# ─────────────────────────────────────────────────────────────────────────────

In [6]:
# 2.3  Assemble the data matrix X ∈ ℝ^(N×9)
# ─────────────────────────────────────────────────────────────────────────────
def generate_baseline_dataset(verbose: bool = True) -> pd.DataFrame:
    """
    Entry point.  Calls all signal generators and returns a tidy DataFrame.
 
    Columns  : time_s, temperature, heart_rate, spo2,
               accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z
    Shape    : (72000, 10)  —  drop time_s before feeding to the model.
    Dtypes   : float64 throughout; cast to float32 for GPU training.
    """
    t = make_time_vector()
 
    temp        = generate_temperature(t, rng)
    hr          = generate_heart_rate(t, rng)
    spo2        = generate_spo2(t, rng)
    ax, ay, az  = generate_accelerometer(t, rng)
    gx, gy, gz  = generate_gyroscope(t, rng)
 
    df = pd.DataFrame({
        'time_s'      : np.round(t,    4),
        'temperature' : np.round(temp, 3),
        'heart_rate'  : np.round(hr,   2),
        'spo2'        : np.round(spo2, 2),
        'accel_x'     : np.round(ax,   4),
        'accel_y'     : np.round(ay,   4),
        'accel_z'     : np.round(az,   4),
        'gyro_x'      : np.round(gx,   3),
        'gyro_y'      : np.round(gy,   3),
        'gyro_z'      : np.round(gz,   3),
    })
 
    if verbose:
        sep = "─" * 60
        print(sep)
        print(f"  Baseline dataset  —  shape: {df.shape}")
        print(f"  Duration : {DURATION / 60:.0f} min  |  fs = {SAMPLE_RATE} Hz")
        print(f"  Seed     : {SEED}")
        print(sep)
        print(df[FEATURE_COLS].describe().round(4).to_string())
        print(sep)
 
    return df
 
 
# ─────────────────────────────────────────────────────────────────────────────

In [7]:
# CLI entry point
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    df = generate_baseline_dataset(verbose=True)
 
    # ── Save CSV (human-readable, includes timestamps)
    csv_path = 'baseline_normal.csv'
    df.to_csv(csv_path, index=False)
    print(f"\nSaved  →  {csv_path}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
 
    # ── Save NumPy array (features only, float32 — ready for torch/keras)
    npy_path = 'baseline_normal.npy'
    X = df[FEATURE_COLS].values.astype(np.float32)
    np.save(npy_path, X)
    print(f"Saved  →  {npy_path}  shape={X.shape}  dtype={X.dtype}")
    print(f"\nNext step: Phase 3 — MinMax / Z-score normalisation + windowing.")

────────────────────────────────────────────────────────────
  Baseline dataset  —  shape: (72000, 10)
  Duration : 60 min  |  fs = 20 Hz
  Seed     : 42
────────────────────────────────────────────────────────────
       temperature  heart_rate        spo2     accel_x     accel_y     accel_z      gyro_x      gyro_y      gyro_z
count   72000.0000  72000.0000  72000.0000  72000.0000  72000.0000  72000.0000  72000.0000  72000.0000  72000.0000
mean       37.0000     71.9971     98.5002     -0.0001      0.0001      0.9999      0.0025      0.0011      0.0013
std         0.1123      3.5907      0.4871      0.0391      0.0392      0.0335      0.9154      0.9162      0.6442
min        36.8080     62.3500     97.5400     -0.1739     -0.1681      0.8460     -4.6480     -3.6980     -2.6480
25%        36.9050     68.9500     98.0600     -0.0263     -0.0265      0.9776     -0.6130     -0.6180     -0.4340
50%        36.9690     71.9900     98.5000     -0.0003      0.0001      0.9999      0.0050     